# 03 - Recurrent, hybrid and attention models

Model shapes, the shared training protocol and the recorded costs.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import torch
from solar_forecasting.models import registry


In [ ]:
n_features, lookback = 33, 24
sizes = {"hidden_size": 32, "n_filters": 8, "kernel_size": 3,
         "attention_size": 16, "d_model": 32, "n_heads": 2,
         "n_layers": 1, "dim_feedforward": 64}
x = torch.zeros(4, lookback, n_features)
for name in ["lstm", "gru", "cnn_lstm", "attention_lstm", "transformer"]:
    model = registry.build_model(name, n_features=n_features, **sizes)
    print(f"{name:>15}: output {tuple(model(x).shape)}")


Every neural model shares one protocol: MAE loss on the standardised
target, early stopping on validation MAE **in watts**, best weights
restored, AdamW, one seed. If the protocol differed per architecture, the
comparison would be a measure of tuning effort rather than of method.


In [ ]:
from solar_forecasting.config import load_config
config = load_config()
print(config.get("training"))
print(config.get("neural"))


In [ ]:
registry = pd.read_csv(ROOT / "results" / "experiments.csv")
neural = registry[registry.model.isin(
    ["lstm", "gru", "cnn_lstm", "attention_lstm", "transformer"])]
display(neural[["model", "horizon", "mae", "rmse", "nrmse_capacity", "r2",
                 "skill_vs_persistence_rmse", "train_seconds",
                 "n_parameters"]].drop_duplicates(["model", "horizon"]))


## What to take from this notebook

- Parameter counts and training durations are recorded per run, so the
  accuracy-cost trade-off is measured rather than asserted.
- At a one-hour horizon, two of these models are worse than persistence.
  That is the result and it is reported as such.
